In [6]:
import pandas as pd
import numpy as np

# Load NAV history
df = pd.read_csv("../data/processed/02_nav_history_cleaned.csv")

# Convert date column
df['date'] = pd.to_datetime(df['date'])

# Sort data
df = df.sort_values(['amfi_code', 'date'])

# Calculate daily returns
df['daily_return'] = df.groupby('amfi_code')['nav'].pct_change()

# Remove null values
df = df.dropna(subset=['daily_return'])

# Compute VaR and CVaR
results = []

for amfi_code, group in df.groupby('amfi_code'):

    returns = group['daily_return']

    var_95 = np.percentile(returns, 5)
    cvar_95 = returns[returns <= var_95].mean()

    results.append({
        'amfi_code': amfi_code,
        'VaR_95': round(var_95, 6),
        'CVaR_95': round(cvar_95, 6)
    })

# Create report
report = pd.DataFrame(results)

# Save report
report.to_csv("var_cvar_report.csv", index=False)

print(report.head())

   amfi_code    VaR_95   CVaR_95
0     100016 -0.014364 -0.018060
1     100025 -0.003793 -0.004994
2     100033 -0.019034 -0.023456
3     101206 -0.013282 -0.017439
4     101207 -0.026021 -0.032459
